In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

MODEL_NAME = "deepseek-chat"

In [2]:
import json
import requests 

def get_weather(city):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,               
        "appid": os.getenv("WEATHER_API_KEY"),
        "units": "metric", 
        "lang":"zh_cn"
    }
    print(os.getenv("WEATHER_API_KEY"))
    response = requests.get(url, params=params)
    data = response.json()
    return json.dumps(data, ensure_ascii=False)

In [3]:
get_weather("Beijing") # Hangzhou  # Shenzhen

7d8e37a126f7a9833d6cb551b94f84b4


'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 800, "main": "Clear", "description": "晴", "icon": "01d"}], "base": "stations", "main": {"temp": 24.94, "feels_like": 23.7, "temp_min": 24.94, "temp_max": 24.94, "pressure": 1015, "humidity": 8, "sea_level": 1015, "grnd_level": 1010}, "visibility": 10000, "wind": {"speed": 5.51, "deg": 345, "gust": 8.99}, "clouds": {"all": 0}, "dt": 1778144259, "sys": {"type": 1, "id": 9609, "country": "CN", "sunrise": 1778101689, "sunset": 1778152428}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'

In [5]:
# 将工具函数封装成符合规范的工具描述
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气情况，一次只能查询一个城市",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，注意：中国的城市需要使用对应城市的英文名称进行代替， 比如：北京的city参数则是Beijing；杭州的city参数则是Hangzhou；深圳的city参数则是Shenzhen",
                    }
                },
                "required": ["city"]
            },
        }
    },
]

In [6]:
messages = [
    {"role": "user", "content": "请帮我查询北京的天气情况"}
]
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
)

In [7]:
response.choices[0].finish_reason

'tool_calls'

In [8]:
tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
import json
function_args = json.loads(tool_call.function.arguments)

print(f"函数名称：{function_name}")
print(f"函数参数：{function_args}")

函数名称：get_weather
函数参数：{'city': 'Beijing'}


In [9]:
available_functions = {
    "get_weather": get_weather,
}

In [10]:
# 具体的函数对象
function_to_call = available_functions[function_name]
function_to_call

<function __main__.get_weather(city)>

In [ ]:
def add(a, b):
    return a + b

add_params = {
    "a": 5,
    "b": 2,
}

add_result = add(**add_params)
print(add_result)

In [12]:
function_response = function_to_call(**function_args)
function_response

7d8e37a126f7a9833d6cb551b94f84b4


'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 800, "main": "Clear", "description": "晴", "icon": "01d"}], "base": "stations", "main": {"temp": 24.94, "feels_like": 23.7, "temp_min": 24.94, "temp_max": 24.94, "pressure": 1015, "humidity": 8, "sea_level": 1015, "grnd_level": 1010}, "visibility": 10000, "wind": {"speed": 5.51, "deg": 345, "gust": 8.99}, "clouds": {"all": 0}, "dt": 1778144259, "sys": {"type": 1, "id": 9609, "country": "CN", "sunrise": 1778101689, "sunset": 1778152428}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'

In [13]:
messages.append(response.choices[0].message.model_dump())
messages.append({
    "role": "tool",   # 固定写法，表示这是工具调用的结果
    "tool_call_id": tool_call.id,   # 关联到具体的工具调用id
    "content": str(function_response)  # 工具执行的结果内容，必须是字符串格式
})

messages

[{'role': 'user', 'content': '请帮我查询北京的天气情况'},
 {'content': '好的，我来帮你查询北京的天气情况。',
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_00_h3fWNcQHhPaCjRfoQIAR1584',
    'function': {'arguments': '{"city": "Beijing"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 0}]},
 {'content': '好的，我来帮你查询北京的天气情况。',
  'refusal': None,
  'role': 'assistant',
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_00_h3fWNcQHhPaCjRfoQIAR1584',
    'function': {'arguments': '{"city": "Beijing"}', 'name': 'get_weather'},
    'type': 'function',
    'index': 0}]},
 {'role': 'tool',
  'tool_call_id': 'call_00_h3fWNcQHhPaCjRfoQIAR1584',
  'content': '{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 800, "main": "Clear", "description": "晴", "icon": "01d"}], "base": "stations", "main": {"temp": 24.94, "feels_like": 23.7, "temp_min": 24.94, "temp_max": 24.94, "

In [14]:
second_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages
)

from IPython.display import Markdown
display(Markdown(second_response.choices[0].message.content))

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. (insufficient tool messages following tool_calls message)", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}